In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist
from sklearn.cluster import KMeans
import pickle
def makefile(what,filename):
    with open(filename,"wb") as f3:
        pickle.dump(what,f3)

def readfile(filename):
    with open(filename,"rb") as f4:
        ans=pickle.load(f4)
    return ans
from math import *

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from PIL import Image
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

In [59]:
dataset=readfile("Xray_pneumonia/train_normal.pkl")
shapes=[]
for sample in dataset:
    shapes.append(sample.shape)    

In [60]:
dataset=readfile("Xray_pneumonia/train_pneumonia.pkl")
shapes=[]
for sample in dataset:
    shapes.append(sample.shape)  

In [64]:
dataset=readfile("Xray_pneumonia/test_pneumonia.pkl")
shapes=[]
for sample in dataset:
    shapes.append(sample.shape)
print(np.array(shapes).min(axis=0))
print(np.array(shapes).max(axis=0))

[344 728]
[1456 2000]


In [65]:
dataset=readfile("Xray_pneumonia/test_normal.pkl")
shapes=[]
for sample in dataset:
    shapes.append(sample.shape)
print(np.array(shapes).min(axis=0))
print(np.array(shapes).max(axis=0))

[496 984]
[2713 2752]


In [7]:
resized=readfile("Xray_pneumonia/train_normal.pkl")

In [14]:
sizes=[2,3,4,5]
step_sizes=[2,3,5,7,10]

train = readfile("Xray_pneumonia/train_normal.pkl")
for ss in step_sizes:
    for size in sizes:
        all_windows=[]
        for n in range(len(train)):
            if n%50==49:
                break
            im_size_i=train[n].shape[0]
            im_size_j=train[n].shape[1]
            
            i_size=0
            i_index=[]
            for i in range(im_size_i-size+1):
                if i %ss ==0:
                    i_index.append(i)
                    i_size+=1
                    
            j_size=0
            j_index=[]
            for j in range(im_size_j-size+1):
                if j %ss ==0:
                    j_index.append(j)
                    j_size+=1
            i_index=np.array(i_index)
            j_index=np.array(j_index)
           
                
            windows=np.zeros([i_size,j_size,size*size])
            position=np.array(range(size**2)).reshape(size,size)
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,num]=train[n][i_index,:][:,j_index]
            windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
            for window in windows:
                #all_windows=np.concatenate([all_windows,windows],axis=0)
                all_windows.append(window)
        all_windows=np.array(all_windows)
        kmeans=KMeans(n_clusters=25)
        kmeans.fit(all_windows)
        LCB=[]
        for n in range(len(train)):
            im_size_i=train[n].shape[0]
            im_size_j=train[n].shape[1]

            i_size=0
            i_index=[]
            for i in range(im_size_i-size+1):
                if i %ss ==0:
                    i_index.append(i)
                    i_size+=1
                    
            j_size=0
            j_index=[]
            for j in range(im_size_j-size+1):
                if j %ss ==0:
                    j_index.append(j)
                    j_size+=1
            i_index=np.array(i_index)
            j_index=np.array(j_index)
           
                
            windows=np.zeros([i_size,j_size,size*size])
            position=np.array(range(size**2)).reshape(size,size)
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,num]=train[n][i_index+i,:][:,j_index+j]
            windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
            cluster=kmeans.predict(windows)
            CB=np.zeros(25)
            for k in range(25):
                CB[k]=len(cluster[cluster==k])
            CB=CB/len(cluster)
            LCB.append(CB)
        LCB=np.array(LCB)
        modelname="KMeans_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+".pkl"
        filename="PLCB_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+".pkl"
        makefile(kmeans,modelname)
        makefile(LCB,filename)
    

In [39]:
sizes=[2,3,4,5]
step_sizes=[10]

train = readfile("Xray_pneumonia/train_normal.pkl")
n=0
for word in train:
    if len(word.shape)==3:
        train[n]=word[:,:,0]
    n+=1
for e in range(5):
    for ss in step_sizes:
        for size in sizes:
            all_windows=[]
            for n in range(len(train)):
                if n%50==49:
                    break
                im_size_i=train[n].shape[0]
                im_size_j=train[n].shape[1]
                
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
               
                    
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=train[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                for window in windows:
                    #all_windows=np.concatenate([all_windows,windows],axis=0)
                    all_windows.append(window)
            all_windows=np.array(all_windows)
            kmeans=KMeans(n_clusters=25)
            kmeans.fit(all_windows)
            LCB=[]
            for n in range(len(train)):
                im_size_i=train[n].shape[0]
                im_size_j=train[n].shape[1]
    
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
               
                    
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=train[n][i_index,:][:,j_index]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB.append(CB)
            LCB=np.array(LCB)
            modelname="KMeans_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            filename="PLCB_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            makefile(kmeans,modelname)
            makefile(LCB,filename)
    

In [ ]:
sizes=[2,3,4,5]
step_sizes=[1]

train = readfile("Xray_pneumonia/train_pneumonia.pkl")
n=0
for word in train:
    if len(word.shape)==3:
        train[n]=word[:,:,0]
    n+=1
for e in range(5):
    for ss in step_sizes:
        for size in sizes:
            all_windows=[]
            for n in range(len(train)):
                if n%50==49:
                    break
                im_size_i=train[n].shape[0]
                im_size_j=train[n].shape[1]
                
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
               
                    
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=train[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                for window in windows:
                    #all_windows=np.concatenate([all_windows,windows],axis=0)
                    all_windows.append(window)
            all_windows=np.array(all_windows)
            kmeans=KMeans(n_clusters=25)
            kmeans.fit(all_windows)
            LCB=[]
            for n in range(len(train)):
                im_size_i=train[n].shape[0]
                im_size_j=train[n].shape[1]
    
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
               
                    
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=train[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB.append(CB)
            LCB=np.array(LCB)
            modelname="KMeans_pneumonia_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            filename="PLCB_pneumonia_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            makefile(kmeans,modelname)
            makefile(LCB,filename)
    

In [ ]:
for e in range(5):
    print(e)
    for ss in step_sizes:
        ens_LCB=0
    
        for size in sizes:
            filename="PLCB_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            LCB=readfile(filename)
            if type(ens_LCB)!=np.ndarray:
                ens_LCB=LCB
            else:
                ens_LCB=np.concatenate([ens_LCB,LCB],axis=1)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        GMM.fit(ens_LCB)
        IF.fit(ens_LCB)
        LOF.fit(ens_LCB)
        OCSVM.fit(ens_LCB)
        LP=ens_LCB.mean(axis=0)
        ens_LCB_test=0
        for size in sizes:
            modelname="KMeans_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            kmeans=readfile(modelname)
            LCB_test=[]
            for n in range(len(resized_test)):
                #im_size=resized_test[n].shape[0]
                im_size_i=resized_test[n].shape[0]
                im_size_j=resized_test[n].shape[1]
    
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=resized_test[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            if type(ens_LCB_test)!=np.ndarray:
                ens_LCB_test=LCB_test
            else:
                ens_LCB_test=np.concatenate([ens_LCB_test,LCB_test],axis=1)
        Unseen_score=(abs(ens_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(ens_LCB_test)
        IF_seen=IF.score_samples(ens_LCB_test)
        LOF_seen=LOF.score_samples(ens_LCB_test)
        OCSVM_seen=OCSVM.score_samples(ens_LCB_test)
        print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
        print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
        print(np.round(roc_auc_score(y,LOF_seen)*100,1))
        print(np.round(roc_auc_score(y,IF_seen)*100,1))
        print(np.round(roc_auc_score(y,GMM_seen)*100,1))

In [ ]:
for e in range(5):
    print(e)
    for ss in step_sizes:
        print(ss)
        for size in sizes:
            modelname="KMeans_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            filename="PLCB_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            kmeans=readfile(modelname)
            LCB=readfile(filename)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            LCB_test=[]
            for n in range(len(resized_test)):
                #im_size=resized_test[n].shape[0]
                im_size_i=resized_test[n].shape[0]
                im_size_j=resized_test[n].shape[1]
                
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=resized_test[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
            print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
            print(np.round(roc_auc_score(y,LOF_seen)*100,1))
            print(np.round(roc_auc_score(y,IF_seen)*100,1))
            print(np.round(roc_auc_score(y,GMM_seen)*100,1))

In [10]:
for e in range(5):
    print(e)
    for ss in step_sizes:
        print(ss)
        for size in sizes:
            modelname="KMeans_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            filename="PLCB_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            kmeans=readfile(modelname)
            LCB=readfile(filename)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            LCB_test=[]
            for n in range(len(resized_test)):
                #im_size=resized_test[n].shape[0]
                im_size_i=resized_test[n].shape[0]
                im_size_j=resized_test[n].shape[1]
                
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=resized_test[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
            print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
            print(np.round(roc_auc_score(y,LOF_seen)*100,1))
            print(np.round(roc_auc_score(y,IF_seen)*100,1))
            print(np.round(roc_auc_score(y,GMM_seen)*100,1))

0
2
60.1
55.5
31.1
48.1
58.3
60.7
56.7
32.1
41.4
59.7
60.9
57.7
32.2
41.9
58.7
60.8
56.3
31.6
43.5
57.8
3
60.7
56.7
30.9
43.9
57.8
60.5
57.7
31.8
38.0
59.3
60.1
57.4
31.1
44.1
58.0
59.6
55.9
30.1
41.7
56.5
5
60.9
55.0
31.9
44.1
60.5
61.5
55.9
32.5
46.8
62.6
60.8
59.0
31.9
37.8
58.1
60.4
55.1
31.1
47.6
58.4
7
61.3
58.4
32.5
42.9
60.5
60.9
57.1
32.1
44.0
59.6
60.9
57.3
32.5
42.7
58.8
61.1
56.9
31.9
40.0
61.5
10
61.6
57.3
33.1
42.0
64.6
60.6
56.0
31.0
41.7
60.6
61.3
56.8
32.3
45.2
61.6
61.5
57.4
33.1
41.7
64.9
1
2
60.0
56.0
31.1
43.3
56.9
61.3
56.0
32.0
39.7
60.9
59.7
55.8
30.1
42.5
57.7
61.0
55.6
31.7
36.6
60.6
3
60.5
55.3
30.9
40.6
59.0
60.8
56.4
32.1
39.2
59.3
61.4
57.9
32.2
41.7
60.8
61.1
56.4
32.2
43.9
61.3
5
61.0
56.4
32.7
45.3
62.4
61.0
56.8
32.2
46.1
61.8
61.1
57.1
32.2
39.7
61.5
60.3
56.5
31.0
39.5
59.3
7
61.4
56.6
33.6
42.8
63.6
60.4
55.7
30.9
42.2
58.9
60.8
58.0
32.8
47.1
61.0
60.0
56.7
30.6
43.1
57.6
10
60.4
57.8
30.6
41.9
59.1
60.4
54.9
29.3
44.8
60.3
61.7
57.0
32.4
43.9
64.2

KeyboardInterrupt: 

In [33]:
for e in range(5):
    print(e)
    for ss in step_sizes:
        #ens_LCB=0
    
        for size in sizes:
            print(size,"x",size)
            filename="PLCB_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            LCB=readfile(filename)
            #if type(ens_LCB)!=np.ndarray:
            #    ens_LCB=LCB
            #else:
            #    ens_LCB=np.concatenate([ens_LCB,LCB],axis=1)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
        #    ens_LCB_test=0
        #for size in sizes:
            modelname="KMeans_normal_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            kmeans=readfile(modelname)
            LCB_test=[]
            for n in range(len(resized_test)):
                #im_size=resized_test[n].shape[0]
                im_size_i=resized_test[n].shape[0]
                im_size_j=resized_test[n].shape[1]
    
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=resized_test[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            #if type(ens_LCB_test)!=np.ndarray:
            #    ens_LCB_test=LCB_test
            #else:
            #    ens_LCB_test=np.concatenate([ens_LCB_test,LCB_test],axis=1)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
            print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
            print(np.round(roc_auc_score(y,LOF_seen)*100,1))
            print(np.round(roc_auc_score(y,IF_seen)*100,1))
            print(np.round(roc_auc_score(y,GMM_seen)*100,1))

0
2 x 2
73.7
72.8
75.9
68.7
75.1
3 x 3
73.4
71.5
75.9
69.8
76.6
4 x 4
73.5
71.3
74.7
71.1
76.0
5 x 5
73.8
71.8
75.4
70.4
76.6
1
2 x 2
73.2
72.8
75.2
66.9
74.6
3 x 3
73.4
72.2
75.4
70.7
74.6
4 x 4
73.4
72.8
75.8
69.8
73.8
5 x 5
73.3
71.6
74.6
72.9
75.0
2
2 x 2
73.3
72.1
73.9
69.7
75.1
3 x 3
73.4
71.8
74.6
68.7
75.2
4 x 4
73.2
72.3
74.8
68.1
75.0
5 x 5
73.2
71.4
75.4
70.0
76.4
3
2 x 2
73.6
72.0
75.4
71.5
75.8
3 x 3
73.5
72.7
75.5
69.8
74.8
4 x 4
73.3
71.0
75.9
69.8
77.0
5 x 5
73.2
71.9
74.5
69.2
74.8
4
2 x 2
72.8
71.9
72.4
66.5
73.5
3 x 3
73.4
73.7
75.4
64.4
73.5
4 x 4
73.0
70.9
71.7
70.2
74.3
5 x 5
73.7
72.5
75.5
68.5
75.7


In [34]:
for e in range(5):
    print(e)
    for ss in step_sizes:
        #ens_LCB=0
    
        for size in sizes:
            print(size,"x",size)
            filename="PLCB_pneumonia_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            LCB=readfile(filename)
            #if type(ens_LCB)!=np.ndarray:
            #    ens_LCB=LCB
            #else:
            #    ens_LCB=np.concatenate([ens_LCB,LCB],axis=1)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
        #    ens_LCB_test=0
        #for size in sizes:
            modelname="KMeans_pneumonia_"+str(ss)+"_"+str(size)+"x"+str(size)+"e"+str(e)+".pkl"
            kmeans=readfile(modelname)
            LCB_test=[]
            for n in range(len(resized_test)):
                #im_size=resized_test[n].shape[0]
                im_size_i=resized_test[n].shape[0]
                im_size_j=resized_test[n].shape[1]
    
                i_size=0
                i_index=[]
                for i in range(im_size_i-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(im_size_j-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                windows=np.zeros([i_size,j_size,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=resized_test[n][i_index+i,:][:,j_index+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(25)
                for k in range(25):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            #if type(ens_LCB_test)!=np.ndarray:
            #    ens_LCB_test=LCB_test
            #else:
            #    ens_LCB_test=np.concatenate([ens_LCB_test,LCB_test],axis=1)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            print(np.round(roc_auc_score(y,Unseen_score*(1))*100,1))
            print(np.round(roc_auc_score(y,-OCSVM_seen)*100,1))
            print(np.round(roc_auc_score(y,-LOF_seen)*100,1))
            print(np.round(roc_auc_score(y,-IF_seen)*100,1))
            print(np.round(roc_auc_score(y,-GMM_seen)*100,1))

0
2 x 2
39.6
41.3
68.5
58.0
42.4
3 x 3
39.6
44.4
67.7
55.0
40.5
4 x 4
39.8
44.6
68.8
59.6
41.8
5 x 5
39.1
41.7
68.3
63.3
40.8
1
2 x 2
39.0
41.5
67.2
59.3
40.3
3 x 3
40.1
42.5
69.2
62.4
43.0
4 x 4
39.1
44.3
67.9
62.7
39.3
5 x 5
38.9
42.4
67.4
60.1
39.1
2
2 x 2
39.4
42.1
68.5
58.0
41.5
3 x 3
39.1
40.7
67.4
57.1
40.3
4 x 4
39.2
43.3
67.3
52.4
39.1
5 x 5
38.8
42.1
67.9
58.1
39.2
3
2 x 2
38.9
42.2
67.6
53.7
39.8
3 x 3
39.1
43.7
67.5
58.4
39.0
4 x 4
38.7
43.8
67.8
54.7
39.1
5 x 5
38.9
43.1
68.1
56.8
39.4
4
2 x 2
39.1
43.5
67.7
62.9
38.8
3 x 3
39.4
41.0
67.6
59.5
40.9
4 x 4
38.9
42.6
67.6
60.8
39.5
5 x 5
39.0
40.9
67.5
55.2
40.3


In [27]:
for size in sizes:
    modelname="KMeans_pneumonia_"+str(size)+"x"+str(size)+".pkl"
    filename="PLCB_pneumonia_"+str(size)+"x"+str(size)+".pkl"
    kmeans=readfile(modelname)
    LCB=readfile(filename)
    GMM=GaussianMixture(n_components=1)
    IF=IsolationForest()
    LOF=LocalOutlierFactor(novelty=True)
    OCSVM=OneClassSVM()
    GMM.fit(LCB)
    IF.fit(LCB)
    LOF.fit(LCB)
    OCSVM.fit(LCB)
    LP=LCB.mean(axis=0)
    LCB_test=[]
    for n in range(len(resized_test)):
        #im_size=resized_test[n].shape[0]
        im_size_i=resized_test[n].shape[0]
        im_size_j=resized_test[n].shape[1]
        windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
        position=np.array(range(size**2)).reshape(size,size)
        for i in range(size):
            for j in range(size):
                num=position[i][j]
                windows[:,:,num]=resized_test[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
        windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
        cluster=kmeans.predict(windows)
        CB=np.zeros(25)
        for k in range(25):
            CB[k]=len(cluster[cluster==k])
        CB=CB/len(cluster)
        LCB_test.append(CB)
    LCB_test=np.array(LCB_test)
    Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
    GMM_seen=GMM.score_samples(LCB_test)
    IF_seen=IF.score_samples(LCB_test)
    LOF_seen=LOF.score_samples(LCB_test)
    OCSVM_seen=OCSVM.score_samples(LCB_test)
    print(np.round(roc_auc_score(y,-Unseen_score*(-1))*100,1))
    print(np.round(roc_auc_score(y,-OCSVM_seen)*100,1))
    print(np.round(roc_auc_score(y,-LOF_seen)*100,1))
    print(np.round(roc_auc_score(y,-IF_seen)*100,1))
    print(np.round(roc_auc_score(y,-GMM_seen)*100,1))

39.1
41.8
67.8
59.6
41.1
38.7
42.0
67.7
52.7
40.3
39.1
41.2
68.0
56.0
41.2
39.1
43.3
67.7
54.8
40.1


In [46]:
ens_LCB=0
for size in sizes:
    filename="PLCB_normal_"+str(size)+"x"+str(size)+".pkl"
    LCB=readfile(filename)
    if type(ens_LCB)!=np.ndarray:
        ens_LCB=LCB
    else:
        ens_LCB=np.concatenate([ens_LCB,LCB],axis=1)
GMM=GaussianMixture(n_components=1)
IF=IsolationForest()
LOF=LocalOutlierFactor(novelty=True)
OCSVM=OneClassSVM()
GMM.fit(ens_LCB)
IF.fit(ens_LCB)
LOF.fit(ens_LCB)
OCSVM.fit(ens_LCB)
LP=ens_LCB.mean(axis=0)
ens_LCB_test=0
for size in sizes:
    modelname="KMeans_normal_"+str(size)+"x"+str(size)+".pkl"
    kmeans=readfile(modelname)
    LCB_test=[]
    for n in range(len(resized_test)):
        #im_size=resized_test[n].shape[0]
        im_size_i=resized_test[n].shape[0]
        im_size_j=resized_test[n].shape[1]
        windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
        position=np.array(range(size**2)).reshape(size,size)
        for i in range(size):
            for j in range(size):
                num=position[i][j]
                windows[:,:,num]=resized_test[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
        windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
        cluster=kmeans.predict(windows)
        CB=np.zeros(25)
        for k in range(25):
            CB[k]=len(cluster[cluster==k])
        CB=CB/len(cluster)
        LCB_test.append(CB)
    LCB_test=np.array(LCB_test)
    if type(ens_LCB_test)!=np.ndarray:
        ens_LCB_test=LCB_test
    else:
        ens_LCB_test=np.concatenate([ens_LCB_test,LCB_test],axis=1)
Unseen_score=(abs(ens_LCB_test-LP)).sum(axis=1)
GMM_seen=GMM.score_samples(ens_LCB_test)
IF_seen=IF.score_samples(ens_LCB_test)
LOF_seen=LOF.score_samples(ens_LCB_test)
OCSVM_seen=OCSVM.score_samples(ens_LCB_test)
print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
print(np.round(roc_auc_score(y,LOF_seen)*100,1))
print(np.round(roc_auc_score(y,IF_seen)*100,1))
print(np.round(roc_auc_score(y,GMM_seen)*100,1))

73.1
72.0
74.1
67.9
74.2


In [49]:
ens_LCB=0
for size in sizes:
    filename="PLCB_pneumonia_"+str(size)+"x"+str(size)+".pkl"
    LCB=readfile(filename)
    if type(ens_LCB)!=np.ndarray:
        ens_LCB=LCB
    else:
        ens_LCB=np.concatenate([ens_LCB,LCB],axis=1)
GMM=GaussianMixture(n_components=1)
IF=IsolationForest()
LOF=LocalOutlierFactor(novelty=True)
OCSVM=OneClassSVM()
GMM.fit(ens_LCB)
IF.fit(ens_LCB)
LOF.fit(ens_LCB)
OCSVM.fit(ens_LCB)
LP=ens_LCB.mean(axis=0)
ens_LCB_test=0
for size in sizes:
    modelname="KMeans_pneumonia_"+str(size)+"x"+str(size)+".pkl"
    kmeans=readfile(modelname)
    LCB_test=[]
    for n in range(len(resized_test)):
        #im_size=resized_test[n].shape[0]
        im_size_i=resized_test[n].shape[0]
        im_size_j=resized_test[n].shape[1]
        windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
        position=np.array(range(size**2)).reshape(size,size)
        for i in range(size):
            for j in range(size):
                num=position[i][j]
                windows[:,:,num]=resized_test[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
        windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
        cluster=kmeans.predict(windows)
        CB=np.zeros(25)
        for k in range(25):
            CB[k]=len(cluster[cluster==k])
        CB=CB/len(cluster)
        LCB_test.append(CB)
    LCB_test=np.array(LCB_test)
    if type(ens_LCB_test)!=np.ndarray:
        ens_LCB_test=LCB_test
    else:
        ens_LCB_test=np.concatenate([ens_LCB_test,LCB_test],axis=1)
Unseen_score=(abs(ens_LCB_test-LP)).sum(axis=1)
GMM_seen=GMM.score_samples(ens_LCB_test)
IF_seen=IF.score_samples(ens_LCB_test)
LOF_seen=LOF.score_samples(ens_LCB_test)
OCSVM_seen=OCSVM.score_samples(ens_LCB_test)
print(np.round(roc_auc_score(y,-Unseen_score*(-1))*100,1))
print(np.round(roc_auc_score(y,-OCSVM_seen)*100,1))
print(np.round(roc_auc_score(y,-LOF_seen)*100,1))
print(np.round(roc_auc_score(y,-IF_seen)*100,1))
print(np.round(roc_auc_score(y,-GMM_seen)*100,1))

38.9
42.1
67.8
52.9
36.7


In [39]:
type(ens_LCB)

numpy.ndarray

In [35]:
type(ens_LCB)

NoneType

In [43]:
ens_LCB.shape

(1341, 100)

In [44]:
ens_LCB

array([[4.37602926e-02, 4.00593969e-02, 4.55668775e-02, ...,
        4.34366173e-02, 2.40669314e-03, 4.53113574e-02],
       [2.94681927e-02, 7.69557963e-02, 4.69117631e-02, ...,
        4.83522297e-02, 2.70461414e-03, 4.33428920e-02],
       [3.32453449e-02, 6.14237134e-02, 5.14847617e-02, ...,
        5.01556326e-02, 3.12287987e-04, 3.72723894e-02],
       ...,
       [3.76922933e-02, 3.21486351e-02, 3.52024165e-02, ...,
        3.37189811e-02, 8.19908440e-03, 3.80255899e-02],
       [3.46114286e-02, 6.91985540e-02, 4.38093750e-02, ...,
        4.10215448e-02, 4.36482995e-06, 2.49432572e-02],
       [1.74430725e-02, 5.79222117e-02, 5.21034821e-02, ...,
        5.25919999e-02, 1.11847757e-03, 3.98062322e-02]])

In [50]:
train = np.array(readfile("Xray_pneumonia/train_normal.pkl"))
size=3
#all_windows=np.zeros([0,size*size])
all_windows=[]
for n in range(len(train)):
    if n%50==49:
        break
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    for window in windows:
        #all_windows=np.concatenate([all_windows,windows],axis=0)
        all_windows.append(window)
all_windows=np.array(all_windows)

In [ ]:
resized=readfile("Xray_pneumonia/train_normal.pkl")

In [8]:
resize_norm=readfile("Xray_pneumonia/test_normal.pkl")
resize_pneu=readfile("Xray_pneumonia/test_pneumonia.pkl")

In [9]:
resized_test=[]
for word in resize_norm:
    resized_test.append(word)
y=np.ones(len(resize_norm))
for word in resize_pneu:
    resized_test.append(word)
y=np.concatenate([y, np.zeros(len(resize_pneu))])    

In [21]:
resized_test[0].shape

(824, 1168)

In [ ]:
train = np.array(readfile("Xray_pneumonia/train_normal.pkl"))
size=7
#all_windows=np.zeros([0,size*size])
all_windows=[]
for n in range(len(train)):
    if n%50==49:
        break
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    for window in windows:
        #all_windows=np.concatenate([all_windows,windows],axis=0)
        all_windows.append(window)
all_windows=np.array(all_windows)

kmeans=KMeans(n_clusters=25)
kmeans.fit(all_windows)
LCB=[]
for n in range(len(train)):
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    cluster=kmeans.predict(windows)
    CB=np.zeros(25)
    for k in range(25):
        CB[k]=len(cluster[cluster==k])
    CB=CB/len(cluster)
    LCB.append(CB)
LCB=np.array(LCB)

In [16]:
windows.shape

(2890370, 16)

In [15]:
LCB

array([[0.04254084, 0.04379686, 0.01496634, ..., 0.04252939, 0.04295093,
        0.04003739],
       [0.05240528, 0.01761782, 0.01035474, ..., 0.02832213, 0.04193953,
        0.02222636],
       [0.05026752, 0.01488602, 0.0454668 , ..., 0.03127409, 0.04722936,
        0.01731566],
       ...,
       [0.03314831, 0.01989902, 0.02345083, ..., 0.03666214, 0.04167311,
        0.02611972],
       [0.0395822 , 0.09333518, 0.00061165, ..., 0.03386735, 0.03638802,
        0.01457769],
       [0.05333331, 0.0308829 , 0.0434415 , ..., 0.01659476, 0.05674844,
        0.01035369]])

In [23]:
makefile(LCB,"normal_train_5x5.pkl")

In [13]:
LCB.shape

(1341, 25)

In [25]:
LCB_test=[]
for n in range(len(resized_test)):
    #im_size=resized_test[n].shape[0]
    im_size_i=resized_test[n].shape[0]
    im_size_j=resized_test[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=resized_test[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    cluster=kmeans.predict(windows)
    CB=np.zeros(25)
    for k in range(25):
        CB[k]=len(cluster[cluster==k])
    CB=CB/len(cluster)
    LCB_test.append(CB)
LCB_test=np.array(LCB_test)
Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
GMM_seen=GMM.score_samples(LCB_test)
IF_seen=IF.score_samples(LCB_test)
LOF_seen=LOF.score_samples(LCB_test)
OCSVM_seen=OCSVM.score_samples(LCB_test)

In [26]:
print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
print(np.round(roc_auc_score(y,LOF_seen)*100,1))
print(np.round(roc_auc_score(y,IF_seen)*100,1))
print(np.round(roc_auc_score(y,GMM_seen)*100,1))

73.1
71.5
74.8
72.3
73.2


In [21]:
LCB_test.shape

(624, 25)

In [20]:
print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
print(np.round(roc_auc_score(y,LOF_seen)*100,1))
print(np.round(roc_auc_score(y,IF_seen)*100,1))
print(np.round(roc_auc_score(y,GMM_seen)*100,1))

73.1
72.0
74.0
67.3
72.6


In [11]:
print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
print(np.round(roc_auc_score(y,LOF_seen)*100,1))
print(np.round(roc_auc_score(y,IF_seen)*100,1))
print(np.round(roc_auc_score(y,GMM_seen)*100,1))

73.2
71.8
74.5
67.8
73.9


In [12]:
windows.shape

(740952, 9)

In [39]:
print(np.round(roc_auc_score(y,Unseen_score*(-1))*100,1))
print(np.round(roc_auc_score(y,OCSVM_seen)*100,1))
print(np.round(roc_auc_score(y,LOF_seen)*100,1))
print(np.round(roc_auc_score(y,IF_seen)*100,1))
print(np.round(roc_auc_score(y,GMM_seen)*100,1))

73.2
72.2
74.1
67.9
73.5


In [6]:
train = np.array(resized)
size=2
all_windows=np.zeros([0,size*size])
for n in range(len(train)):
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    all_windows=np.concatenate([all_windows,windows],axis=0)
kmeans=KMeans(n_clusters=25)
kmeans.fit(all_windows)
LCB=[]
for n in range(len(train)):
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    cluster=kmeans.predict(windows)
    CB=np.zeros(25)
    for k in range(25):
        CB[k]=len(cluster[cluster==k])
    CB=CB/len(cluster)
    LCB.append(CB)
LCB=np.array(LCB)

KeyboardInterrupt: 

In [16]:
train = np.array(readfile("Xray_pneumonia/train_pneumonia.pkl"))
n=0
for word in train:
    if len(word.shape)==3:
        train[n]=word[:,:,0]
    n+=1
size=5
#all_windows=np.zeros([0,size*size])
all_windows=[]
for n in range(len(train)):
    if n%50==49:
        break
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    for window in windows:
        #all_windows=np.concatenate([all_windows,windows],axis=0)
        all_windows.append(window)
all_windows=np.array(all_windows)

kmeans=KMeans(n_clusters=25)
kmeans.fit(all_windows)
LCB=[]
for n in range(len(train)):
    im_size_i=train[n].shape[0]
    im_size_j=train[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=train[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    cluster=kmeans.predict(windows)
    CB=np.zeros(25)
    for k in range(25):
        CB[k]=len(cluster[cluster==k])
    CB=CB/len(cluster)
    LCB.append(CB)
LCB=np.array(LCB)

In [17]:
GMM=GaussianMixture(n_components=1)
IF=IsolationForest()
LOF=LocalOutlierFactor(novelty=True)
OCSVM=OneClassSVM()
GMM.fit(LCB)
IF.fit(LCB)
LOF.fit(LCB)
OCSVM.fit(LCB)
LP=LCB.mean(axis=0)

In [18]:
LCB_test=[]
for n in range(len(resized_test)):
    #im_size=resized_test[n].shape[0]
    im_size_i=resized_test[n].shape[0]
    im_size_j=resized_test[n].shape[1]
    windows=np.zeros([im_size_i-size+1,im_size_j-size+1,size*size])
    position=np.array(range(size**2)).reshape(size,size)
    for i in range(size):
        for j in range(size):
            num=position[i][j]
            windows[:,:,num]=resized_test[n][i:im_size_i-size+1+i,j:im_size_j-size+1+j]
    windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
    cluster=kmeans.predict(windows)
    CB=np.zeros(25)
    for k in range(25):
        CB[k]=len(cluster[cluster==k])
    CB=CB/len(cluster)
    LCB_test.append(CB)
LCB_test=np.array(LCB_test)
Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
GMM_seen=GMM.score_samples(LCB_test)
IF_seen=IF.score_samples(LCB_test)
LOF_seen=LOF.score_samples(LCB_test)
OCSVM_seen=OCSVM.score_samples(LCB_test)

In [ ]:
for size in sizes:
    modelname=file
    

In [ ]:
resized=[]